# Preprocessing — Getting the Data Model-Ready

Every decision here has a reason. This notebook documents what I did, what I tried first, and why I changed course.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle, warnings, os

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
os.makedirs('../outputs/figures', exist_ok=True)
os.makedirs('../data/processed', exist_ok=True)

df = pd.read_csv('../data/raw/credit_risk.csv')
print(f'Raw: {df.shape}  |  Missing: {df.isnull().sum().sum()} cells')

## Step 1 — Drop Two Features Before Doing Anything Else

**`loan_percent_income`** — this column equals `loan_amnt / person_income`. Since `loan_amnt` is our target, this is direct data leakage. If we kept it, the model would simply reconstruct the target from income, producing artificially perfect results that fall apart on real data.

**`loan_status`** — whether the loan was eventually repaid or defaulted. This information only exists *after* the loan is issued and the repayment period ends. At the time you'd actually want to predict the loan amount (i.e., at application), this value doesn't exist yet.

In [ ]:
print('Columns before dropping:', list(df.columns))
df_model = df.drop(columns=['loan_percent_income', 'loan_status'])
print('Columns after dropping: ', list(df_model.columns))
print(f'Dropped 2 columns. Shape: {df_model.shape}')

## Step 2 — Impute Missing Values

### What the original code did (wrong)
```python
credit.dropna(axis=1, inplace=True)  # deletes entire COLUMNS
```
This deleted `loan_int_rate` and `person_emp_length` from the feature set entirely.

### What I do instead
- **`person_emp_length`** (2.7% missing): global median. Missing employment length often means self-employed or new workforce entry — no strong directional signal, so median is conservative.
- **`loan_int_rate`** (9.6% missing): per-grade median. Interest rate is set by the lender based on loan grade, so a grade-A loan will never be missing a rate in the same range as a grade-G loan. Group-level imputation is more principled than a global fill.

In [ ]:
# Show the grade-level interest rate medians — this is why group imputation makes sense
grade_medians = df_model.groupby('loan_grade')['loan_int_rate'].median().sort_index()
print('Interest rate median by grade:')
print(grade_medians.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
grade_medians.plot(kind='bar', ax=axes[0], color='#C62828', edgecolor='white')
axes[0].set_title('Per-grade int_rate medians span 8%–21% — global fill would be wrong', fontweight='bold')
axes[0].set_xlabel('Loan Grade'); axes[0].set_ylabel('Median Rate (%)')
axes[0].tick_params(axis='x', rotation=0)

df_model['person_emp_length'].hist(bins=30, ax=axes[1], color='#1565C0', edgecolor='white')
axes[1].axvline(df_model['person_emp_length'].median(), color='red', lw=2, linestyle='--',
                label=f'Median = {df_model["person_emp_length"].median()}')
axes[1].set_title('Employment length — median imputation is safe here', fontweight='bold')
axes[1].legend()
plt.tight_layout()
plt.savefig('../outputs/figures/imputation_rationale.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
df_model['person_emp_length'] = df_model['person_emp_length'].fillna(df_model['person_emp_length'].median())
df_model['loan_int_rate']     = df_model['loan_int_rate'].fillna(
    df_model.groupby('loan_grade')['loan_int_rate'].transform('median'))

assert df_model.isnull().sum().sum() == 0
print('No missing values remain. Shape unchanged:', df_model.shape)

## Step 3 — Encode Categorical Features

I use one-hot encoding instead of label encoding. `loan_intent` categories (MEDICAL, PERSONAL, EDUCATION…) have no ordinal relationship — there's no meaningful sense in which MEDICAL > EDUCATION numerically. One-hot avoids imposing a false ordering.

Same logic for `person_home_ownership`, `loan_grade`, and `cb_person_default_on_file`.

In [ ]:
cat_cols = ['person_home_ownership', 'loan_intent', 'loan_grade', 'cb_person_default_on_file']
df_enc = pd.get_dummies(df_model, columns=cat_cols, drop_first=True)

y = df_enc['loan_amnt']
X = df_enc.drop(columns=['loan_amnt'])

print(f'Features: {X.shape[1]} (up from 9 after one-hot encoding)')
print(f'Feature list:')
for f in X.columns: print(f'  {f}')

## Step 4 — Train / Test Split

80/20 random split. No stratification needed for regression targets. `random_state=42` for reproducibility.

Note: if this were time-series loan data with dates, a temporal split would be mandatory. This dataset has no timestamps, so random split is appropriate.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print(f'Train target mean: ${y_train.mean():,.0f}')
print(f'Test  target mean: ${y_test.mean():,.0f}  (should be close to train)')

## Step 5 — Scale Features AND Target

### Why scale X?
Income ranges from $4,000 to $6,000,000. Without scaling, early gradient updates are dominated by income and training destabilises. StandardScaler brings all features to mean≈0, std≈1.

### Why also scale y?
Loan amounts range $500–$35,000. For a neural network with MSE loss, unscaled targets produce loss values in the billions (e.g., MSE of a $6,000 error = 36,000,000). Gradients become enormous and training diverges. Scaling y to mean≈0, std≈1 stabilises this completely.

**Critical**: inverse-transform predictions before showing them to users. A scaled prediction of 1.3 means nothing — you need to convert it back to ~$21,000.

In [ ]:
scaler_X = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train)
X_test_sc  = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train.values.reshape(-1,1)).flatten()
y_test_sc  = scaler_y.transform(y_test.values.reshape(-1,1)).flatten()

print(f'y_train scaled — mean: {y_train_sc.mean():.3f}, std: {y_train_sc.std():.3f}')
print(f'X_train feature means (should be ~0): min={X_train_sc.mean(axis=0).min():.3f}, max={X_train_sc.mean(axis=0).max():.3f}')

# Demonstrate inverse transform
example_scaled = np.array([[1.3]])
example_dollars = scaler_y.inverse_transform(example_scaled)
print(f'\nExample: scaled output 1.3 → ${example_dollars[0][0]:,.0f} in loan amount')

## Step 6 — Save Processed Data and Scalers

In [ ]:
train_df = pd.DataFrame(X_train_sc, columns=X.columns)
train_df['loan_amnt_scaled'] = y_train_sc
train_df.to_csv('../data/processed/train.csv', index=False)

test_df = pd.DataFrame(X_test_sc, columns=X.columns)
test_df['loan_amnt_scaled'] = y_test_sc
test_df['loan_amnt_raw']    = y_test.values
test_df.to_csv('../data/processed/test.csv', index=False)

with open('../data/processed/scaler_X.pkl', 'wb') as f: pickle.dump(scaler_X, f)
with open('../data/processed/scaler_y.pkl', 'wb') as f: pickle.dump(scaler_y, f)
with open('../data/processed/feature_names.pkl', 'wb') as f: pickle.dump(list(X.columns), f)

print('Saved:')
print(f'  data/processed/train.csv       ({train_df.shape[0]:,} rows, {train_df.shape[1]} cols)')
print(f'  data/processed/test.csv        ({test_df.shape[0]:,} rows)')
print(f'  data/processed/scaler_X.pkl')
print(f'  data/processed/scaler_y.pkl    ← needed to inverse-transform NN predictions')

## Preprocessing Summary

| Step | Decision | Why |
|------|----------|-----|
| Drop `loan_percent_income` | Excluded entirely | = loan_amnt/income — direct target leakage |
| Drop `loan_status` | Excluded entirely | Post-issuance outcome, not available at application time |
| Impute `loan_int_rate` | Per-grade median | Rate is grade-dependent; global median adds noise |
| Impute `person_emp_length` | Global median | No directional signal for direction of fill |
| Encode categoricals | One-hot | No ordinal relationship in intent, ownership, grade categories |
| Scale X | StandardScaler | Income range $4k–$6M would destabilise NN training |
| Scale y | StandardScaler | Unscaled MSE in billions causes gradient divergence |